<a href="https://colab.research.google.com/github/thanakppm/Project_Python_Retail_Store_Group_4/blob/main/Retail_Store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Python Retail Store Group 4


```
683020241-0 นางสาวณัฏฐ์ปรัศมน ทวีโภควรกุล
683020250-9 นางสาวนรมณ พวงศรีเคน
683020563-8 นางสาวกฤษณา กาลิง
683020572-7 นายธนากร พงษ์เพชร
683020589-0 นางสาวพัชริญา ภูโนนทา
```

In [ ]:
# ============================================================
# ระบบจัดการร้านค้าปลีก (Retail Store Management System)
# ============================================================
# ร้านค้าปลีกมีสินค้าอยู่ 2 จุด คือ
# 1. Shelf      = สินค้าที่วางขายให้ลูกค้า
# 2. Warehouse  = สินค้าสำรองหลังร้าน
# เมื่อลูกค้าซื้อสินค้า:
# Customer
#     ↓
# Order
#     ↓
# Payment
#     ↓
# ตัดสินค้าออกจาก Shelf
#     ↓
# ตรวจสอบ Stock
#     ↓
# แจ้งเตือนถ้าสินค้าใกล้หมด
# ============================================================


# ============================================================
# Class: Customer
# ============================================================

import datetime

class Customer:

    def __init__(self, customer_id, name, is_member=False, points=0):
        self.customer_id = customer_id
        self.name = name
        self.is_member = is_member
        self.points = points

    def add_points(self, points):
        # Method นี้ใช้เพิ่มคะแนนให้ลูกค้า
        # โดยจะเพิ่มคะแนนก็ต่อเมื่อเป็นสมาชิกเท่านั้น
        if self.is_member:
            self.points += points

    def customer_info(self):
        # Method นี้ใช้แสดงข้อมูลของลูกค้า
        return {
            "customer_id": self.customer_id,
            "name": self.name,
            "is_member": self.is_member,
            "points": self.points
        }


# ============================================================
# Class: Product
# ============================================================

class Product:

    def __init__(self, product_id, name, category, cost, price):
        self.product_id = product_id
        self.name = name
        self.category = category
        self.cost = cost
        self.price = price

    def get_profit_per_unit(self):
        # คำนวณกำไรต่อสินค้า 1 ชิ้น
        # กำไรต่อหน่วย = ราคาขาย - ต้นทุน
        return self.price - self.cost

    def product_info(self):
        # แสดงข้อมูลของสินค้า
        return {
            "product_id": self.product_id,
            "name": self.name,
            "category": self.category,
            "cost": self.cost,
            "price": self.price,
            "profit": self.get_profit_per_unit()
        }


# ============================================================
# Class: Inventory
# ============================================================
#
# ใช้จัดการ Stock ของสินค้าใน 2 จุด:
#
# Shelf      = สินค้าที่พร้อมขาย
# Warehouse  = สินค้าสำรอง
#
# ระบบจะแยก Stock สองจุดออกจากกัน เพื่อให้เจ้าของร้านรู้ว่า
# สินค้าหมดที่ชั้นวาง หรือสินค้าสำรองในโกดังก็ใกล้หมดแล้ว
# ============================================================

class Inventory:

    def __init__(self):
        # เก็บข้อมูล Stock ของสินค้าแต่ละชนิด
        self.stock = {}

    def add_product(
        self,
        product,
        shelf=0,
        warehouse=0,
        shelf_min=5,              # จุดที่เริ่มเตือนให้เติมสินค้าบนชั้น
        warehouse_min=10     # warehouse_min  = จุดที่เริ่มเตือนให้สั่งสินค้าเพิ่ม
    ):

        self.stock[product.product_id] = {
            "product": product,
            "shelf": shelf,
            "warehouse": warehouse,
            "shelf_min": shelf_min,
            "warehouse_min": warehouse_min
        }

    def restock_shelf(self, product, quantity):
        # Method นี้ใช้ย้ายสินค้าจาก Warehouse
        # มาเติมที่ Shelf
        # สินค้าไม่ได้เพิ่มขึ้นจากการเติมชั้น
        # แต่เป็นการย้ายตำแหน่งของ Stock

        if quantity <= 0:
            print("จำนวนสินค้าต้องมากกว่า 0")
            return False

        if product.product_id not in self.stock:
            print("ไม่พบสินค้าใน Inventory")
            return False

        item = self.stock[product.product_id]

        # ตรวจสอบก่อนว่า Warehouse มีของเพียงพอหรือไม่
        if item["warehouse"] < quantity:
            print("สินค้าในโกดังไม่เพียงพอ")
            return False

        # ย้ายสินค้าจาก Warehouse มาที่ Shelf
        item["warehouse"] -= quantity
        item["shelf"] += quantity
        print(
            f"เติม {product.name} จำนวน {quantity} ชิ้น "
            f"จาก Warehouse → Shelf"
        )

        return True

    def sell_product(self, product, quantity):
        # Method นี้ใช้ตัดสินค้าออกจาก Shelf
        # เมื่อมีการชำระเงินเรียบร้อยแล้ว

        if quantity <= 0:
            print("จำนวนสินค้าต้องมากกว่า 0")
            return False

        if product.product_id not in self.stock:
            print("ไม่พบสินค้าใน Inventory")
            return False

        item = self.stock[product.product_id]

        # ตรวจสอบว่าสินค้าบน Shelf เพียงพอหรือไม่
        if item["shelf"] < quantity:
            print(f"สินค้า {product.name} บนชั้นวางไม่เพียงพอ")
            return False

        # ตัด Stock ออกจาก Shelf
        item["shelf"] -= quantity

        return True

    def can_sell(self, product, quantity):
        # ตรวจสอบ Stock ก่อนการขายจริง
        # Method นี้สำคัญสำหรับป้องกันกรณีที่ Order มีหลายสินค้า
        # แล้วสินค้าบางรายการมีไม่เพียงพอ

        if quantity <= 0:
            return False

        if product.product_id not in self.stock:
            return False

        return self.stock[product.product_id]["shelf"] >= quantity

    def adjust_stock(self, product, shelf=None, warehouse=None):
        # ใช้ปรับ Stock กรณีตรวจนับสินค้าจริงแล้ว
        # พบว่าจำนวนในระบบไม่ตรงกับจำนวนที่มีอยู่จริง
        #
        # เช่น ระบบบันทึกว่ามี 20 ชิ้น
        # แต่ตรวจนับจริงพบว่ามี 18 ชิ้น

        if product.product_id not in self.stock:
            print("ไม่พบสินค้าใน Inventory")
            return False

        item = self.stock[product.product_id]

        # ตรวจสอบไม่ให้ Stock ติดลบ
        if shelf is not None and shelf < 0:
            print("จำนวนสินค้า Shelf ไม่สามารถติดลบได้")
            return False

        if warehouse is not None and warehouse < 0:
            print("จำนวนสินค้า Warehouse ไม่สามารถติดลบได้")
            return False

        # ปรับเฉพาะค่าที่ผู้ใช้ส่งเข้ามา
        if shelf is not None:
            item["shelf"] = shelf

        if warehouse is not None:
            item["warehouse"] = warehouse

        return True

    def check_alert(self):
        # ตรวจสอบสินค้าที่มี Stock ต่ำกว่าจุดที่กำหนด
        # ระบบนี้ช่วยลดภาระเจ้าของร้าน
        # เพราะไม่ต้องเดินตรวจสินค้าเองทุกจุด

        alerts = []

        for item in self.stock.values():

            product = item["product"]

            # ถ้าสินค้าบน Shelf ต่ำกว่าหรือเท่ากับจุดขั้นต่ำ
            # ระบบจะแจ้งเตือนให้เติมสินค้า
            if item["shelf"] <= item["shelf_min"]:
                alerts.append(
                    f"⚠️ {product.name}: "
                    f"Shelf เหลือ {item['shelf']} ชิ้น "
                    f"ควรเติมสินค้า"
                )

            # ถ้าสินค้าใน Warehouse ต่ำกว่าหรือเท่ากับจุดขั้นต่ำ
            # ระบบจะแจ้งเตือนให้สั่งสินค้าเพิ่ม
            if item["warehouse"] <= item["warehouse_min"]:
                alerts.append(
                    f"⚠️ {product.name}: "
                    f"Warehouse เหลือ {item['warehouse']} ชิ้น "
                    f"ควรสั่งสินค้าเพิ่ม"
                )

        return alerts

    def receive_stock(self, product, quantity):
        # Method นี้ใช้จำลองกรณีร้านได้รับสินค้าใหม่
        # จากนั้นสินค้าจะถูกนำเข้า Warehouse

        if quantity <= 0:
            print("จำนวนสินค้าต้องมากกว่า 0")
            return False

        if product.product_id not in self.stock:
            print("ไม่พบสินค้าใน Inventory")
            return False

        self.stock[product.product_id]["warehouse"] += quantity

        print(
            f"รับสินค้า {product.name} "
            f"เข้า Warehouse จำนวน {quantity} ชิ้น"
        )

        return True

    def show_stock(self, product):
        # แสดงจำนวน Stock ของสินค้าทั้ง 2 จุด

        if product.product_id not in self.stock:
            print("ไม่พบสินค้าใน Inventory")
            return

        item = self.stock[product.product_id]

        print(f"\nสินค้า: {product.name}")
        print(f"Shelf: {item['shelf']} ชิ้น")
        print(f"Warehouse: {item['warehouse']} ชิ้น")


# ============================================================
# Class: Order
# ============================================================
# Class นี้แทน "รายการซื้อของลูกค้า 1 ครั้ง"
# หนึ่ง Order สามารถมีสินค้าหลายชนิดได้
# ตอน add_product() ยังไม่ตัด Stock
# เพราะลูกค้าอาจเปลี่ยนใจก่อนชำระเงิน
# Stock จะถูกตัดเมื่อ pay() สำเร็จ
# ============================================================

class Order:
    def __init__(self, order_id, customer, inventory, order_time=None):
        self.order_id = order_id
        self.customer = customer
        self.inventory = inventory
        self.items = []
        self.total = 0
        self.is_paid = False
        self.order_time = order_time

    def add_product(self, product, quantity):

        if self.is_paid:
            print("Order นี้ชำระเงินแล้ว ไม่สามารถเพิ่มสินค้าได้")
            return False

        if quantity <= 0:
            print("จำนวนสินค้าต้องมากกว่า 0")
            return False

        # ตรวจสอบว่า Product มีอยู่ใน Inventory หรือไม่
        if product.product_id not in self.inventory.stock:
            print("ไม่พบสินค้าใน Inventory")
            return False

        # ตรวจสอบว่าสินค้าชนิดเดียวกันถูกเพิ่มใน Order ไปแล้วหรือไม่
        current_quantity = 0

        for item in self.items:
            if item["product"].product_id == product.product_id:
                current_quantity += item["quantity"]

        # ตรวจสอบ Stock รวมของสินค้าชนิดนี้
        # ทั้งหมดที่ลูกค้าต้องการใน Order นี้ต้องไม่เกิน Shelf
        if not self.inventory.can_sell(
            product,
            current_quantity + quantity
        ):
            print(
                f"สินค้า {product.name} บน Shelf "
                f"ไม่เพียงพอสำหรับ Order นี้"
            )
            return False

        # ถ้ามีสินค้าชนิดนี้อยู่แล้ว ให้เพิ่มจำนวน
        for item in self.items:

            if item["product"].product_id == product.product_id:

                item["quantity"] += quantity

                self.total += product.price * quantity

                return True

        # ถ้ายังไม่มีสินค้าใน Order ให้สร้างรายการใหม่
        self.items.append({
            "product": product,
            "quantity": quantity
        })

        # เพิ่มราคาสินค้าเข้าไปในยอดรวม
        self.total += product.price * quantity

        return True

    def remove_product(self, product):
        # ลบสินค้าออกจาก Order
        # สามารถทำได้เฉพาะก่อนชำระเงิน

        if self.is_paid:
            print("Order นี้ชำระเงินแล้ว ไม่สามารถแก้ไขได้")
            return False

        for item in self.items:

            if item["product"].product_id == product.product_id:

                # หักราคาของสินค้านั้นออกจากยอดรวม
                self.total -= (
                    item["product"].price * item["quantity"]
                )

                # ลบสินค้าออกจากรายการ
                self.items.remove(item)

                return True

        print("ไม่พบสินค้านี้ใน Order")
        return False

    def pay(self):
        # Method นี้ใช้ชำระเงิน
        #
        # ลำดับการทำงาน:
        #
        # 1. ตรวจสอบว่า Order ยังไม่ได้จ่าย
        # 2. ตรวจสอบ Stock ของสินค้าทุกตัวก่อน
        # 3. ถ้าทุกอย่างมี Stock เพียงพอ จึงตัด Stock
        # 4. เปลี่ยนสถานะเป็นชำระเงินแล้ว
        #
        # การตรวจสอบทั้งหมดก่อนตัด Stock
        # ช่วยป้องกันปัญหา "ตัด Stock ไปบางรายการแล้ว
        # แต่รายการถัดไปขายไม่ได้"

        if self.is_paid:
            print("Order นี้ชำระเงินไปแล้ว")
            return False

        if len(self.items) == 0:
            print("Order ไม่มีสินค้า")
            return False

        # ----------------------------------------------------
        # ขั้นที่ 1: ตรวจสอบ Stock ของสินค้าทุกตัวก่อน
        # ----------------------------------------------------

        for item in self.items:

            product = item["product"]
            quantity = item["quantity"]

            if not self.inventory.can_sell(product, quantity):

                print(
                    f"ไม่สามารถชำระเงินได้ "
                    f"เพราะสินค้า {product.name} ไม่เพียงพอ"
                )

                # ยังไม่มีการตัด Stock
                return False

        # ----------------------------------------------------
        # ขั้นที่ 2: เมื่อทุกสินค้าผ่านการตรวจสอบแล้ว
        # จึงค่อยตัด Stock จริง
        # ----------------------------------------------------

        for item in self.items:

            product = item["product"]
            quantity = item["quantity"]

            self.inventory.sell_product(
                product,
                quantity
            )

        # เปลี่ยนสถานะเป็นชำระเงินแล้ว
        self.is_paid = True

        # คำนวณคะแนนสมาชิก
        # ตัวอย่างเงื่อนไขของระบบจำลอง:
        # ทุก 50 บาท = 1 คะแนน
        points = int(self.total // 50)

        self.customer.add_points(points)

        print(
            f"ชำระเงิน Order {self.order_id} สำเร็จ "
            f"ยอดรวม {self.total:.2f} บาท"
        )

        return True

    def show_bill(self):
        # แสดงใบเสร็จของ Order

        print("\n" + "=" * 40)
        print(f"Order ID: {self.order_id}")
        print(f"Customer: {self.customer.name}")
        print("=" * 40)

        for item in self.items:

            product = item["product"]
            quantity = item["quantity"]

            subtotal = product.price * quantity

            print(
                f"{product.name} x {quantity} "
                f"= {subtotal:.2f} บาท"
            )

        print("-" * 40)
        print(f"Total: {self.total:.2f} บาท")

        if self.is_paid:
            print("สถานะ: ชำระเงินแล้ว")
        else:
            print("สถานะ: ยังไม่ชำระเงิน")

        print("=" * 40)


# ============================================================
# Class: DailyReport
# ============================================================
# Class นี้ใช้สร้างรายงานยอดขายของ "แต่ละวัน"
#
# ระบบจะรับวันที่ที่ต้องการดู เช่น "2026-08-29"
# แล้วเลือกเฉพาะ Order ที่:
# 1. ชำระเงินแล้ว
# 2. เกิดขึ้นในวันที่ที่เราต้องการ
#
# ทำให้ DailyReport เป็นรายงานของวันนั้นจริง ๆ
# ไม่ใช่การนำ Order ทุกวันมารวมกัน
# ============================================================

class DailyReport:

    def __init__(self, orders, report_date):

        # เก็บรายการ Order ทั้งหมด
        self.orders = orders

        # เก็บวันที่ที่ต้องการสร้างรายงาน
        # ใช้รูปแบบ YYYY-MM-DD
        self.report_date = report_date

    def get_paid_orders(self):
        # เลือกเฉพาะ Order ที่ชำระเงินแล้ว
        # และเกิดขึ้นในวันที่ต้องการรายงาน

        paid_orders = []

        for order in self.orders:

            # ตรวจสอบว่า Order มีวันเวลาหรือไม่
            if order.order_time is None:
                continue

            # แปลงวันเวลาของ Order ให้เป็นวันที่
            # เพื่อใช้เปรียบเทียบกับ report_date
            order_date = order.order_time.strftime("%Y-%m-%d")

            # ต้องเป็น Order ที่ชำระเงินแล้ว
            # และเกิดขึ้นในวันที่ต้องการ
            if order.is_paid and order_date == self.report_date:
                paid_orders.append(order)

        return paid_orders

    def summary(self):
        # สรุปข้อมูลการขายของวันที่กำหนด

        paid_orders = self.get_paid_orders()

        # คำนวณยอดขายรวม
        total_sales = sum(
            order.total
            for order in paid_orders
        )

        # เตรียมตัวแปรสำหรับคำนวณต้นทุน
        total_cost = 0

        # เตรียมตัวแปรสำหรับนับจำนวนสินค้าที่ขาย
        total_items = 0

        # วนดู Order ของวันที่ต้องการทีละรายการ
        for order in paid_orders:

            # ในแต่ละ Order อาจมีสินค้าหลายชนิด
            for item in order.items:

                product = item["product"]
                quantity = item["quantity"]

                # ต้นทุนรวมของสินค้านั้น
                total_cost += product.cost * quantity

                # จำนวนสินค้าที่ขายทั้งหมด
                total_items += quantity

        # กำไร = ยอดขาย - ต้นทุน
        total_profit = total_sales - total_cost

        return {
            "report_date": self.report_date,
            "total_orders": len(paid_orders),
            "total_items": total_items,
            "total_sales": total_sales,
            "total_cost": total_cost,
            "total_profit": total_profit
        }

    def show_summary(self):
        # แสดงรายงานยอดขายของวันที่กำหนด

        result = self.summary()

        print("\n" + "=" * 40)
        print(f"รายงานยอดขายประจำวันที่ {result['report_date']}")
        print("=" * 40)

        print(
            f"จำนวน Order: "
            f"{result['total_orders']}"
        )

        print(
            f"จำนวนสินค้าที่ขาย: "
            f"{result['total_items']} ชิ้น"
        )

        print(
            f"ยอดขายรวม: "
            f"{result['total_sales']:.2f} บาท"
        )

        print(
            f"ต้นทุนรวม: "
            f"{result['total_cost']:.2f} บาท"
        )

        print(
            f"กำไรรวม: "
            f"{result['total_profit']:.2f} บาท"
        )

        print("=" * 40)
